# Bronze load — GA4 sample ecommerce events

Python 3.12 notebook (not Spark). Default lakehouse: **GA4Lakehouse** in workspace **GA4-Report**.

Reads `Files/raw/ga4/ingest_date=2026-11-12/events/event_date=YYYYMMDD.parquet`, writes one Delta table with overwrite then per-day append, adds `_ingest_date`, unifies nested schemas across days, and checks the Delta row count against the source parquets.

**Schema decision:** this lakehouse is schema-enabled. A custom `bronze` schema needs `CREATE SCHEMA` via Spark SQL or the SQL endpoint, which a Python notebook cannot run. The table therefore goes in the default schema `dbo` as `bronze_ga4_events`. Nested GA4 columns (event_params, user_properties, items, device, geo, traffic_source, ecommerce, etc.) are kept as structs and lists.

Physical path: `/lakehouse/default/Tables/dbo/bronze_ga4_events` (or the equivalent `abfss://` URI, which Python notebooks need for Delta commits — the FUSE mount cannot rename).

Days are written one at a time so the notebook does not hold the full ~4.3M-row dataset in memory. Partitioned by `event_date` (one partition per day).


In [ ]:
# Load daily GA4 event parquets into dbo.bronze_ga4_events (overwrite + append).
import os
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import DeltaTable, write_deltalake

ROOT = Path(os.environ.get("LAKEHOUSE_DEFAULT", "/lakehouse/default"))
INGEST_DATE = "2026-11-12"
EVENTS_DIR = ROOT / "Files/raw/ga4" / f"ingest_date={INGEST_DATE}" / "events"
TABLE = "bronze_ga4_events"
# BigQuery table metadata totals (get_table num_rows), not a query result.
EXPECTED_DAYS = 92
EXPECTED_ROWS = 4_295_584


def _storage():
    """abfss + bearer token in Fabric; local filesystem path otherwise."""
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        ws = ctx["defaultLakehouseWorkspaceId"]
        lh = ctx["defaultLakehouseId"]
        endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
        base = f"abfss://{ws}@{endpoint}/{lh}/Tables/dbo"
        opts = {
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        }
        return base, opts
    except Exception:
        dbo = ROOT / "Tables" / "dbo"
        dbo.mkdir(parents=True, exist_ok=True)
        return str(dbo), None


def _delta_rows(uri, storage_options):
    return DeltaTable(uri, storage_options=storage_options).to_pyarrow_dataset().count_rows()


def _cast_null_type(typ: pa.DataType) -> pa.DataType:
    if pa.types.is_null(typ):
        return pa.string()
    if pa.types.is_struct(typ):
        return pa.struct([
            pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
            for f in typ
        ])
    if pa.types.is_list(typ) or pa.types.is_large_list(typ) or pa.types.is_fixed_size_list(typ):
        return pa.list_(_cast_null_type(typ.value_type))
    if pa.types.is_map(typ):
        return pa.map_(_cast_null_type(typ.key_type), _cast_null_type(typ.item_type))
    return typ


def _cast_nulls(table: pa.Table) -> pa.Table:
    new_fields = []
    new_cols = []
    for i, field in enumerate(table.schema):
        new_type = _cast_null_type(field.type)
        col = table.column(i)
        if new_type != field.type:
            col = col.cast(new_type)
        new_fields.append(pa.field(field.name, new_type, nullable=field.nullable))
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))


def _unify(paths: list[Path]) -> pa.Schema:
    schemas = [pq.read_schema(p) for p in paths]
    try:
        unified = pa.unify_schemas(schemas, promote_options="permissive")
    except TypeError:
        unified = pa.unify_schemas(schemas)
    fields = [
        pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
        for f in unified
    ]
    names = [f.name for f in fields]
    if "_ingest_date" not in names:
        fields.append(pa.field("_ingest_date", pa.string()))
    return pa.schema(fields)


def _align(table: pa.Table, schema: pa.Schema) -> pa.Table:
    target_names = [f.name for f in schema if f.name != "_ingest_date"]
    arrays = []
    fields = []
    for name in target_names:
        field = schema.field(name)
        if name in table.schema.names:
            col = table.column(name)
            if col.type != field.type:
                col = col.cast(field.type)
        else:
            col = pa.nulls(table.num_rows, type=field.type)
        arrays.append(col)
        fields.append(field)
    return pa.Table.from_arrays(arrays, schema=pa.schema(fields))


files = sorted(EVENTS_DIR.glob("event_date=*.parquet"))
if not files:
    raise SystemExit(f"FAIL no parquet files under {EVENTS_DIR}")

schema = _unify(files)
base, opts = _storage()
uri = f"{base}/{TABLE}"
print(f"ingest_date={INGEST_DATE}")
print(f"write target: {uri}  (schema dbo, bronze_ prefix)")
print(f"source files: {len(files)}  unified fields: {len(schema)}")

src_total = 0
results = []
for i, src in enumerate(files):
    try:
        table = pq.read_table(src)
        src_n = table.num_rows
        src_total += src_n
        table = _align(table, schema)
        ingest = pa.array([INGEST_DATE] * src_n, type=pa.string())
        table = table.append_column("_ingest_date", ingest)
        table = _cast_nulls(table)
        if i == 0:
            write_deltalake(
                uri,
                table,
                mode="overwrite",
                schema_mode="overwrite",
                partition_by=["event_date"],
                storage_options=opts,
            )
        else:
            write_deltalake(
                uri,
                table,
                mode="append",
                partition_by=["event_date"],
                storage_options=opts,
            )
        print(f"ok   {src.name}: rows={src_n:,}")
        results.append(True)
    except Exception as e:
        print(f"FAIL {src.name}: {type(e).__name__}: {e}")
        results.append(False)

got = _delta_rows(uri, opts)
notes = []
ok = all(results) and results and got == src_total
if got != src_total:
    notes.append(f"delta {got} != source {src_total}")
if len(files) != EXPECTED_DAYS:
    notes.append(f"files {len(files)} != BQ metadata days {EXPECTED_DAYS}")
if src_total != EXPECTED_ROWS:
    notes.append(
        f"source {src_total} != BQ metadata rows {EXPECTED_ROWS} "
        "(skipped as a hard fail; dry-run or new export)"
    )
status = "PASS" if ok else "FAIL"
extra = f"  {'; '.join(notes)}" if notes else ""
print(f"{status} {TABLE}: delta={got:,} source={src_total:,} days={len(files)}{extra}")
